# Phase 3 RR: Hierarchical, context-aware Rhetorical Roles on the Phase-1 Transformer-XL + GQA model

This notebook takes the **same idea used in the GQLA hierarchical notebook** — classify each sentence
*in context* instead of in isolation — and applies it to the **Transformer-XL + GQA** backbone from
`phase3_rr_transformer_xl_finetuning_6experiments.ipynb`.

**What changed vs. the original Transformer-XL notebook (which plateaued around Macro-F1 ≈ 0.25–0.27):**

| Problem in the original notebook | Fix here |
|---|---|
| Each sentence classified **in isolation** from its own last-token state only. | **Hierarchical model**: Transformer-XL+GQA encodes each sentence → a **2-layer BiLSTM over the whole document** (+ relative-position features + domain embedding) → label per sentence. |
| Heavy over-fitting (only 80 training documents total). | Parameter-efficient adaptation (LoRA / LoRMA / Prefix / Prompt), feature dropout, class-balanced loss, early stopping on dev. |
| Rare classes (Issue, Dissent, PrecedentOverruled, None) swamped by plain cross-entropy. | Square-root inverse-frequency class weights + optional prior-correction / Viterbi transition smoothing at decode time. |
| Macro-F1 computed over all 13 labels even when a label never occurs in dev (counts as 0, capping the score). | Report **both** sklearn-default macro-F1 (labels present in gold ∪ predicted) and the strict 13-label version. |
| One model, one run. | 6 required experiments + an ensemble over all of them. |

**Experiments (all start from your Phase-1 Transformer-XL + GQA checkpoint, same as the original notebook):**

| # | Experiment key | Technique |
|---|---|---|
| 1 | `full_finetune` | Full fine-tuning of the Phase-1 backbone + BiLSTM head |
| 2 | `lora` | LoRA (Hu et al., 2021) on `q_proj,k_proj,v_proj,out_proj` |
| 3 | `lorma` | **LoRMA+** (Bihany, Patel, Modi, ACL 2025) — multiplicative low-rank update (compulsory) |
| 4 | `prefix_tuning` | Prefix Tuning (Li & Liang, 2021) — reuses the Transformer-XL `memory` argument |
| 5 | `prompt_tuning` | Prompt Tuning (Lester et al., 2021) — learnable soft-prompt tokens prepended per sentence |
| 6 | `knowledge_distillation` | Compact layer-strided student distilled from the `full_finetune` teacher (compulsory) |

Everything downstream of the per-sentence encoder (the BiLSTM document head, class weights, windowed
training, decoding, ensembling) is carried over from the GQLA hierarchical notebook essentially unchanged;
only the per-sentence **encoder** (tokenizer + backbone + adapters) is the Transformer-XL + GQA code from
your original Phase-3 Transformer-XL notebook.

**This notebook has not been run end-to-end against a live checkpoint/GPU.** Macro-F1 > 0.60 is a
**target**, not a guarantee — with ~80 training documents total, dev-set noise alone is roughly ±0.05.
Section 13 explains what to look at and what to try next if you land below it.

## 1. Setup and configuration

In [ ]:
import os, re, json, math, random, unicodedata, gc, time, copy, pickle, itertools, zipfile
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.optim import AdamW
from sklearn.metrics import f1_score, classification_report, confusion_matrix

try:
    from google.colab import drive
    drive.mount('/content/drive'); IN_COLAB = True
except Exception:
    IN_COLAB = False

SEED = 42
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE)

# ---------------- paths: EDIT THESE to match your Phase-1 Transformer-XL + GQA run ----------------
PHASE1_WORK_DIR  = "/content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/transformer_xl_gqa_checkpoints"
CHECKPOINT_PATH  = os.path.join(PHASE1_WORK_DIR, "best_model_transformer_xl_gqa-small.pt")
TOKENIZER_PATH   = "/content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/final_bpe_en.json"
OUTPUT_DIR       = "/content/drive/MyDrive/CS781/Capstone-Project/Phase-3-TXL-hier"
DATA_DIR         = "./phase3_rr_data"
PROB_DIR         = os.path.join(OUTPUT_DIR, "probs")
for d in (OUTPUT_DIR, DATA_DIR, PROB_DIR): os.makedirs(d, exist_ok=True)

# ---------------- HF token: DO NOT hard-code it ----------------
# Colab: add a secret named HF_TOKEN (key icon in the left sidebar), or export HF_TOKEN in the environment.
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")

# ---------------- Must match the Phase-1 Transformer-XL + GQA run exactly ----------------
MODEL_CONFIG = {
    "vocab_size": 32768,
    "d_model": 768,
    "num_layers": 8,
    "num_query_heads": 32,
    "num_kv_heads": 8,
    "memory_length": 128,
    "dropout": 0.1,
}

# ---------------- model / training hyper-parameters ----------------
MAX_TOK         = 128    # max BPE tokens kept per sentence
WINDOW          = 32     # consecutive sentences per training window (BiLSTM sees windows in training, whole docs at test time)
WINDOWS_PER_BATCH = 2
ENC_MICRO_TRAIN = 32     # sentences pushed through the backbone at once while training
ENC_MICRO_EVAL  = 96
HEAD_LR         = 2e-3
HEAD_WD         = 0.01
LABEL_SMOOTH    = 0.02
GRAD_CLIP       = 1.0
PATIENCE        = 5
N_POS_BINS      = 20
N_POS_F         = 6
FORCE_RERUN     = False  # True = ignore saved probabilities and retrain everything

# Per-technique hyperparameters (Section 9.2.1 of the capstone document). Only the adaptation
# *mechanism* differs across experiments -- see Section 5 for the implementations.
LORA_R, LORA_ALPHA   = 16, 32
LORMA_R, LORMA_ALPHA = 16, 16
PREFIX_LEN = 32
PROMPT_LEN = 32
STUDENT_LAYER_STRIDE = 4   # knowledge-distillation student keeps every 4th teacher block
DISTILL_T  = 2.0
DISTILL_ALPHA = 0.7

EXPERIMENTS = {
    "full_finetune":          dict(method="full",   lr=3e-5, epochs=10),
    "lora":                   dict(method="lora",   lr=3e-4, epochs=12, r=LORA_R, alpha=LORA_ALPHA),
    "lorma":                  dict(method="lorma",  lr=3e-4, epochs=12, r=LORMA_R, alpha=LORMA_ALPHA),
    "prefix_tuning":          dict(method="prefix", lr=5e-4, epochs=12, prefix_len=PREFIX_LEN),
    "prompt_tuning":          dict(method="prompt", lr=5e-4, epochs=12, prompt_len=PROMPT_LEN),
    "knowledge_distillation": dict(method="full",   lr=8e-5, epochs=14, student_stride=STUDENT_LAYER_STRIDE,
                                    kd_T=DISTILL_T, kd_alpha=DISTILL_ALPHA),
}

Mounted at /content/drive
Device: cuda


## 2. Download and clean the official Phase-3 data
Same source as the original notebook. Unlike that notebook's cleaning step, **no sentence is ever
dropped** here (empty ones become `"."`), so the number of predictions always equals the number of
test sentences -- a hard requirement of the submission format (Section 9.3.5 of the capstone document).

In [ ]:
from huggingface_hub import hf_hub_download
REPO_ID = "Exploration-Lab/CS-781-Capstone"
PHASE3_FILES = {
    "CL_train": "Phase_3_RR/Train/CL_train.json", "CL_dev": "Phase_3_RR/Train/CL_dev.json",
    "IT_train": "Phase_3_RR/Train/IT_train.json", "IT_dev": "Phase_3_RR/Train/IT_dev.json",
    "label_vocab": "Phase_3_RR/Train/label_vocab.json",
    "CL_test": "Phase_3_RR/Test/CL_test.json", "IT_test": "Phase_3_RR/Test/IT_test.json",
}
def read_json(p):
    with open(p, encoding="utf-8") as f: return json.load(f)

raw = {}
for key, fn in PHASE3_FILES.items():
    local = os.path.join(DATA_DIR, fn)
    if not os.path.isfile(local):
        local = hf_hub_download(repo_id=REPO_ID, filename=fn, repo_type="dataset", token=HF_TOKEN, local_dir=DATA_DIR)
    raw[key] = read_json(local)

def get_labels(v):
    if isinstance(v, list): labs = v
    elif isinstance(v, dict) and all(isinstance(x, int) for x in v.values()):
        labs = [k for k, _ in sorted(v.items(), key=lambda kv: kv[1])]
    elif isinstance(v, dict) and isinstance(v.get("labels"), list): labs = v["labels"]
    else: labs = list(v.keys())
    return [str(x) for x in labs]
LABELS = get_labels(raw["label_vocab"]); K = len(LABELS)
label_to_id = {l: i for i, l in enumerate(LABELS)}; id_to_label = {i: l for l, i in label_to_id.items()}
print("Labels:", LABELS)

def normalize_sentence(t):
    t = "" if t is None else unicodedata.normalize("NFKC", str(t)).replace(" ", " ")
    t = re.sub(r"\s+", " ", t).strip()
    return t if t else "."          # never drop a sentence: output length must equal input length

def to_rows(obj, labeled, name):
    recs = [{"id": k, "text": v} for k, v in obj.items()] if isinstance(obj, dict) else obj
    rows = []; seen = set()
    for i, r in enumerate(recs):
        did = str(r.get("id", r.get("doc_id", i)))
        text = r.get("text", r.get("sentences"))
        if isinstance(text, str): text = [text]
        assert did not in seen, f"{name}: duplicate id {did}"; seen.add(did)
        row = {"id": did, "text": [normalize_sentence(s) for s in text]}
        if labeled:
            ys = [str(y) for y in r["labels"]]
            assert len(ys) == len(row["text"]), f"{name}/{did}: sentence/label mismatch"
            bad = [y for y in ys if y not in label_to_id]; assert not bad, f"unknown labels {bad[:3]}"
            row["labels"] = ys
        rows.append(row)
    return rows

cleaned = {k: to_rows(raw[k], not k.endswith("test"), k) for k in PHASE3_FILES if k != "label_vocab"}
for k, rows in cleaned.items():
    print(f"{k:9s} docs={len(rows):3d} sentences={sum(len(r['text']) for r in rows):6d}")
for k in ["CL_train", "IT_train", "CL_dev", "IT_dev"]:
    c = Counter(y for r in cleaned[k] for y in r["labels"])
    print(k, {l: c.get(l, 0) for l in LABELS})

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

Labels: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']
CL_train  docs= 40 sentences= 10593
CL_dev    docs=  5 sentences=  1396
IT_train  docs= 40 sentences=  6252
IT_dev    docs=  5 sentences=   746
CL_test   docs=  5 sentences=  1339
IT_test   docs=  5 sentences=   858
CL_train {'Fact': 2979, 'Issue': 17, 'ArgumentPetitioner': 976, 'ArgumentRespondent': 996, 'Statute': 367, 'Dissent': 284, 'PrecedentReliedUpon': 1305, 'PrecedentNotReliedUpon': 209, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 438, 'RatioOfTheDecision': 2856, 'RulingByPresentCourt': 113, 'None': 53}
IT_train {'Fact': 1119, 'Issue': 90, 'ArgumentPetitioner': 637, 'ArgumentRespondent': 352, 'Statute': 732, 'Dissent': 0, 'PrecedentReliedUpon': 640, 'PrecedentNotReliedUpon': 360, 'PrecedentOverruled': 3, 'RulingByLowerCourt': 436, 'RatioOfT

## 3. Tokenizer (identical to your Phase-1 BPE tokenizer)

In [ ]:
import os, re, json, heapq, collections

try:
    import psutil
    _PROC = psutil.Process(os.getpid())
except Exception:
    _PROC = None

def _ram_gb():
    return _PROC.memory_info().rss / (1024 ** 3) if _PROC else 0.0

WORD_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)
_WORKER = None
def _init_worker(tok):
    global _WORKER
    _WORKER = tok
def _encode_chunk(args):
    texts, add_special = args
    return [_WORKER.encode(t, add_special) for t in texts]

class BPETokenizer:
    _HEAP_THRESHOLD = 24
    _CACHE_LIMIT = 500_000

    def __init__(self, vocab_size, end_marker="</w>", reserved_tokens=None):
        self.vocab_size = vocab_size
        self.end_marker = end_marker
        self.reserved_tokens = reserved_tokens if reserved_tokens is not None else ["<pad>", "<unk>", "<s>", "</s>"]
        self.itos = list(self.reserved_tokens)
        self.stoi = {t: i for i, t in enumerate(self.reserved_tokens)}
        self.merges = {}
        self._ready = False
        self._cache = {}

    @staticmethod
    def from_json(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = BPETokenizer(vocab_size=data["vocab_size"], end_marker=data.get("end_marker", "</w>"),
                            reserved_tokens=data.get("reserved_tokens", ["<pad>", "<unk>", "<s>", "</s>"]))
        tok.itos = data["itos"]; tok.stoi = data["stoi"]
        tok.merges = {tuple(p): r for p, r in data["merges"]}
        tok._ready = False; tok._cache = {}
        return tok

    def _prepare_inference(self):
        if self._ready: return
        self._byte_ids = [self.stoi[bytes([b]).decode("latin-1")] for b in range(256)]
        self._first_merge_id = len(self.itos) - len(self.merges)
        self._end_id = self.stoi.get(self.end_marker, self.stoi["<unk>"])
        self._bos = self.stoi["<s>"]; self._eos = self.stoi["</s>"]
        base = self._first_merge_id
        for (a, b), r in self.merges.items():
            if self.itos[base + r] != self.itos[a] + self.itos[b]:
                raise ValueError("Merge/vocab mismatch: cannot infer merge ids from ranks.")
        self._M = len(self.itos); M = self._M
        self._ranks = {a * M + b: r for (a, b), r in self.merges.items()}
        self._cache = {}; self._ready = True

    def _merge_small(self, ids):
        get, M, base = self._ranks.get, self._M, self._first_merge_id
        while len(ids) > 1:
            best, best_i = None, -1
            for i in range(len(ids) - 1):
                r = get(ids[i] * M + ids[i + 1])
                if r is not None and (best is None or r < best):
                    best, best_i = r, i
            if best is None: break
            a, b, new = ids[best_i], ids[best_i + 1], base + best
            out, i, n = [], 0, len(ids)
            while i < n:
                if i < n - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(new); i += 2
                else:
                    out.append(ids[i]); i += 1
            ids = out
        return ids

    def _merge_heap(self, ids):
        get, M, base = self._ranks.get, self._M, self._first_merge_id
        n = len(ids)
        nxt = list(range(1, n + 1)); nxt[-1] = -1
        prv = list(range(-1, n - 1)); alive = [True] * n
        heap = []
        for i in range(n - 1):
            r = get(ids[i] * M + ids[i + 1])
            if r is not None: heap.append((r, i))
        heapq.heapify(heap)
        while heap:
            r, i = heapq.heappop(heap)
            if not alive[i]: continue
            j = nxt[i]
            if j == -1 or get(ids[i] * M + ids[j]) != r: continue
            ids[i] = base + r; alive[j] = False
            k = nxt[j]; nxt[i] = k
            if k != -1: prv[k] = i
            p = prv[i]
            if p != -1:
                r2 = get(ids[p] * M + ids[i])
                if r2 is not None: heapq.heappush(heap, (r2, p))
            if k != -1:
                r2 = get(ids[i] * M + ids[k])
                if r2 is not None: heapq.heappush(heap, (r2, i))
        return [ids[i] for i in range(n) if alive[i]]

    def _bpe_word(self, word):
        hit = self._cache.get(word)
        if hit is not None: return hit
        ids = [self._byte_ids[b] for b in word.encode("utf-8")]
        ids.append(self._end_id)
        ids = self._merge_small(ids) if len(ids) <= self._HEAP_THRESHOLD else self._merge_heap(ids)
        result = tuple(ids)
        if len(self._cache) < self._CACHE_LIMIT: self._cache[word] = result
        return result

    def encode(self, text, add_special_tokens=True):
        self._prepare_inference()
        out = [self._bos] if add_special_tokens else []
        extend = out.extend; cache_get = self._cache.get; bpe = self._bpe_word
        for w in WORD_RE.findall(text):
            t = cache_get(w)
            if t is None: t = bpe(w)
            extend(t)
        if add_special_tokens: out.append(self._eos)
        return out

    def decode(self, token_ids):
        self._prepare_inference()
        filtered_ids = [tid for tid in token_ids if self.itos[tid] not in self.reserved_tokens]
        decoded_text = "".join([self.itos[tid] for tid in filtered_ids]).replace(self.end_marker, " ")
        return decoded_text.strip()

In [ ]:
if not os.path.isfile(TOKENIZER_PATH): raise FileNotFoundError(TOKENIZER_PATH)
tokenizer = BPETokenizer.from_json(TOKENIZER_PATH)
PAD_ID = tokenizer.stoi.get("<pad>", 0); BOS_ID = tokenizer.stoi.get("<s>", 2); EOS_ID = tokenizer.stoi.get("</s>", 3)
VOCAB_SIZE = len(tokenizer.itos)
assert VOCAB_SIZE == MODEL_CONFIG["vocab_size"], "tokenizer vocab must match the Phase-1 checkpoint"
print("Tokenizer vocab:", VOCAB_SIZE)

Tokenizer vocab: 32768


## 4. Transformer-XL + GQA architecture (identical to your Phase-1 / original Phase-3 notebook)

In [ ]:
import math

class TransformerXLRelativePositionalEncoding(nn.Module):
    def __init__(self, d_model: int, base: float = 10000.0):
        super().__init__()
        if d_model % 2:
            raise ValueError("d_model must be even")
        self.d_model = d_model
        inv_freq = 1.0 / (base ** (torch.arange(0, d_model, 2, dtype=torch.float32) / d_model))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, query_length: int, key_length: int, device=None):
        if key_length <= 0:
            raise ValueError("key_length must be positive")
        if device is None:
            device = self.inv_freq.device
        pos_seq = torch.arange(key_length - 1, -1, -1, dtype=torch.float32, device=device)
        sinusoid_inp = torch.outer(pos_seq, self.inv_freq.to(device))
        return torch.cat([sinusoid_inp.sin(), sinusoid_inp.cos()], dim=-1)

class TransformerXLGQAAttention(nn.Module):
    def __init__(self, d_model, num_query_heads, num_kv_heads, dropout=0.0):
        super().__init__()
        if d_model % num_query_heads != 0:
            raise ValueError("d_model must be divisible by num_query_heads")
        if num_query_heads % num_kv_heads != 0:
            raise ValueError("num_query_heads must be divisible by num_kv_heads")
        self.d_model = d_model
        self.num_query_heads = num_query_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = d_model // num_query_heads
        self.group_size = num_query_heads // num_kv_heads

        self.q_proj = nn.Linear(d_model, num_query_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.r_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.u = nn.Parameter(torch.zeros(num_query_heads, self.head_dim))
        self.v = nn.Parameter(torch.zeros(num_query_heads, self.head_dim))
        nn.init.normal_(self.u, std=0.02)
        nn.init.normal_(self.v, std=0.02)
        self.dropout = nn.Dropout(dropout)

    def _q_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.num_query_heads, self.head_dim).transpose(1, 2)

    def _kv_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

    def _expand_kv(self, x):
        B, H, T, D = x.shape
        return x.unsqueeze(2).expand(B, H, self.group_size, T, D).reshape(B, self.num_query_heads, T, D)

    @staticmethod
    def _rel_shift(x):
        B, H, Q, K = x.shape
        zero = torch.zeros((B, H, Q, 1), dtype=x.dtype, device=x.device)
        x = torch.cat([zero, x], dim=-1)
        x = x.view(B, H, K + 1, Q)
        x = x[:, :, 1:].view(B, H, Q, K)
        return x

    @staticmethod
    def _causal_mask(query_length, key_length, device):
        memory_length = key_length - query_length
        return torch.triu(torch.ones((query_length, key_length), dtype=torch.bool, device=device),
                           diagonal=memory_length + 1)

    def forward(self, x, memory=None, relative_position=None):
        B, Q, _ = x.shape
        if memory is not None and memory.size(1) > 0:
            kv_input = torch.cat([memory, x], dim=1)
        else:
            kv_input = x
        K = kv_input.size(1)

        q = self._q_heads(self.q_proj(x))
        k = self._expand_kv(self._kv_heads(self.k_proj(kv_input)))
        v = self._expand_kv(self._kv_heads(self.v_proj(kv_input)))

        ac = torch.einsum("bhqd, bhkd->bhqk", q, k)
        ac = ac + torch.einsum("hd,bhkd->bhk", self.u, k).unsqueeze(2)

        if relative_position is None:
            raise ValueError("relative_position is required for Transformer-XL attention")

        r = self._expand_kv(self._kv_heads(self.r_proj(relative_position).unsqueeze(0)))
        bd = torch.einsum("bhqd,bhkd->bhqk", q, r)
        bd = bd + torch.einsum("hd,bhkd->bhk", self.v, r).unsqueeze(2)
        bd = self._rel_shift(bd)

        scores = (ac + bd) / math.sqrt(self.head_dim)
        mask = self._causal_mask(Q, K, x.device)
        scores = scores.masked_fill(mask.view(1, 1, Q, K), torch.finfo(scores.dtype).min)
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, Q, self.d_model)
        return self.out_proj(out)


class TransformerXLBlockGQA(nn.Module):
    def __init__(self, d_model, num_query_heads, num_kv_heads, dropout=0.0, ffn_mult=4):
        super().__init__()
        self.attn = TransformerXLGQAAttention(d_model, num_query_heads, num_kv_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        ffn_dim = ffn_mult * d_model
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model), nn.Dropout(dropout),
        )

    def forward(self, x, memory, relative_position):
        x = self.norm1(x + self.attn(x, memory=memory, relative_position=relative_position))
        x = self.norm2(x + self.ffn(x))
        return x


class TransformerXLGQAModel(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_query_heads, num_kv_heads, memory_length,
                 dropout=0.0, tie_embeddings=True):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.num_layers = num_layers
        self.memory_length = memory_length
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = TransformerXLRelativePositionalEncoding(d_model)
        self.blocks = nn.ModuleList([
            TransformerXLBlockGQA(d_model, num_query_heads, num_kv_heads, dropout) for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        if tie_embeddings:
            self.lm_head.weight = self.embedding.weight

    def forward(self, input_ids, memory=None, labels=None):
        x = self.embedding(input_ids)
        Q = x.size(1)
        if memory is None:
            memory = [None] * self.num_layers
        if len(memory) != self.num_layers:
            raise ValueError(f"memory must contain {self.num_layers} entries")
        mem_len = 0 if memory[0] is None else memory[0].size(1)
        K = mem_len + Q
        rel_pos = self.pos_encoding(Q, K, device=x.device)
        for i, block in enumerate(self.blocks):
            x = block(x, memory[i], rel_pos)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.vocab_size), labels.reshape(-1))
        return {"logits": logits, "loss": loss}

In [ ]:
if not os.path.isfile(CHECKPOINT_PATH): raise FileNotFoundError(CHECKPOINT_PATH)

def build_phase1_model():
    m = TransformerXLGQAModel(
        vocab_size=MODEL_CONFIG["vocab_size"], d_model=MODEL_CONFIG["d_model"],
        num_layers=MODEL_CONFIG["num_layers"], num_query_heads=MODEL_CONFIG["num_query_heads"],
        num_kv_heads=MODEL_CONFIG["num_kv_heads"], memory_length=MODEL_CONFIG["memory_length"],
        dropout=MODEL_CONFIG["dropout"],
    )
    ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    state = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
    missing, unexpected = m.load_state_dict(state, strict=False)
    if missing or unexpected:
        print("Missing:", missing[:10], "count=", len(missing))
        print("Unexpected:", unexpected[:10], "count=", len(unexpected))
        raise RuntimeError("Checkpoint is not compatible with the configured Transformer-XL architecture.")
    return m

BASE_BACKBONE = build_phase1_model()
del BASE_BACKBONE.lm_head            # LM head is not used for classification
D_MODEL = MODEL_CONFIG["d_model"]; FEAT_DIM = 2 * D_MODEL   # mean-pool + last-token pool, concatenated
print("Phase-1 Transformer-XL + GQA loaded (encoder only): %.2fM params" %
      (sum(p.numel() for p in BASE_BACKBONE.parameters()) / 1e6))

Phase-1 Transformer-XL + GQA loaded (encoder only): 75.96M params


## 5. Document tensors, position features, class weights, metrics

In [ ]:
def pos_features(n, lens):
    i = np.arange(n, dtype=np.float32); rel = i / max(n - 1, 1)
    f = np.stack([rel, 1 - rel, np.log1p(i) / 6, np.log1p(n - 1 - i) / 6,
                  np.full(n, np.log1p(n) / 6, np.float32), np.minimum(lens, MAX_TOK) / MAX_TOK], 1).astype(np.float32)
    bins = np.minimum((rel * N_POS_BINS).astype(np.int64), N_POS_BINS - 1)
    return f, bins

TOK_CACHE = {}
def tok_ids(s):
    t = TOK_CACHE.get(s)
    if t is None:
        t = tokenizer.encode(s, add_special_tokens=True)
        if len(t) > MAX_TOK: t = t[:MAX_TOK - 1] + [EOS_ID]
        TOK_CACHE[s] = t
    return t

def build_docs(rows, dom, labeled):
    out = []
    for r in rows:
        ids = [tok_ids(s) for s in r["text"]]; n = len(ids)
        posf, posb = pos_features(n, np.array([len(x) for x in ids]))
        y = np.array([label_to_id[l] for l in r["labels"]], dtype=np.int64) if labeled else None
        out.append(dict(id=r["id"], domain=dom, n=n, ids=ids, posf=posf, posb=posb, y=y, sents=r["text"]))
    return out
t0 = time.time()
DOCS = {k: build_docs(v, 0 if k.startswith("CL") else 1, not k.endswith("test")) for k, v in cleaned.items()}
SPLITS = list(DOCS)
print("tokenised in %.0fs" % (time.time() - t0), {k: len(v) for k, v in DOCS.items()})
TRAIN_DOCS = DOCS["CL_train"] + DOCS["IT_train"]

# class weights: sqrt inverse frequency, absent classes get weight 0
counts = np.zeros(K)
for d in TRAIN_DOCS: counts += np.bincount(d["y"], minlength=K)
w = (counts.sum() / (K * np.maximum(counts, 1))) ** 0.5; w[counts == 0] = 0
w = w / ((w * counts).sum() / counts.sum())
CLASS_W = torch.tensor(w, dtype=torch.float32, device=DEVICE)
print("class weights:", {LABELS[i]: round(float(w[i]), 2) for i in range(K)})

# per-domain prior and transition matrices (for decoding)
PRIOR = np.maximum(counts, 30) / np.maximum(counts, 30).sum()
TRANS = {}
for dom, split in ((0, "CL_train"), (1, "IT_train")):
    C = np.full((K, K), 0.5)
    for d in DOCS[split]:
        for a, b in zip(d["y"][:-1], d["y"][1:]): C[a, b] += 1
    TRANS[dom] = np.log(C / C.sum(1, keepdims=True))

def f1_variants(yt, yp):
    return (f1_score(yt, yp, average="macro", zero_division=0),
            f1_score(yt, yp, labels=list(range(K)), average="macro", zero_division=0))

def eval_dev(probs, decode=None):
    res = {}; allt, allp = [], []
    for dom in ("CL", "IT"):
        yt, yp = [], []
        for d in DOCS[dom + "_dev"]:
            p = probs[dom + "_dev"][d["id"]]
            yp.append(p.argmax(1) if decode is None else decode(p, d["domain"])); yt.append(d["y"])
        yt, yp = np.concatenate(yt), np.concatenate(yp); allt.append(yt); allp.append(yp)
        res[dom], res[dom + "_13"] = f1_variants(yt, yp)
    res["mean"] = (res["CL"] + res["IT"]) / 2; res["mean_13"] = (res["CL_13"] + res["IT_13"]) / 2
    res["pooled"], res["pooled_13"] = f1_variants(np.concatenate(allt), np.concatenate(allp))
    return res

tokenised in 1s {'CL_train': 40, 'CL_dev': 5, 'IT_train': 40, 'IT_dev': 5, 'CL_test': 5, 'IT_test': 5}
class weights: {'Fact': 0.66, 'Issue': 4.06, 'ArgumentPetitioner': 1.05, 'ArgumentRespondent': 1.14, 'Statute': 1.27, 'Dissent': 2.49, 'PrecedentReliedUpon': 0.95, 'PrecedentNotReliedUpon': 1.76, 'PrecedentOverruled': 24.26, 'RulingByLowerCourt': 1.42, 'RatioOfTheDecision': 0.64, 'RulingByPresentCourt': 2.4, 'None': 2.44}


## 6. Adaptation methods: Full FT, LoRA, **LoRMA**, Prefix Tuning, Prompt Tuning

Same adapter implementations as the original Transformer-XL notebook (Section 7 there), just wired up
through a single `apply_method` dispatcher like the GQLA hierarchical notebook uses. LoRA/LoRMA are
applied to the four attention projections (`q_proj, k_proj, v_proj, out_proj`) of every
`TransformerXLBlockGQA`. Prefix tuning reuses the model's own `memory` argument (a per-layer learnable
"virtual memory" prepended to keys/values -- exactly what Transformer-XL's memory mechanism is for).
Prompt tuning prepends learnable soft-prompt embeddings to the token embeddings before the first block.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, r=16, alpha=32):
        super().__init__(); self.base = base
        for p in self.base.parameters(): p.requires_grad = False
        d_out, d_in = base.weight.shape
        self.r = r; self.scaling = alpha / r
        self.lora_A = nn.Parameter(torch.randn(r, d_in) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(d_out, r))
    def forward(self, x):
        return self.base(x) + self.scaling * ((x @ self.lora_A.t()) @ self.lora_B.t())


class LoRMALinear(nn.Module):
    """LoRMA+ (Bihany, Patel, Modi; ACL 2025 Findings; arXiv:2506.07621): multiplicative low-rank
    update of the frozen pretrained weight, W' = (I + (alpha/r) B A) @ W0. A is zero-initialised so
    training starts exactly at the identity transform (W' = W0)."""
    def __init__(self, base, r=16, alpha=16):
        super().__init__(); self.base = base
        for p in self.base.parameters(): p.requires_grad = False
        d_out, d_in = base.weight.shape
        self.r = r; self.scaling = alpha / r
        self.lorma_B = nn.Parameter(torch.randn(d_out, r) * 0.01)
        self.lorma_A = nn.Parameter(torch.zeros(r, d_out))
        self.register_buffer("_eye", torch.eye(d_out), persistent=False)
    def effective_weight(self):
        inflation = self._eye + self.scaling * (self.lorma_B @ self.lorma_A)
        return inflation @ self.base.weight
    def forward(self, x):
        return F.linear(x, self.effective_weight(), self.base.bias)


ADAPTER_TARGET_PROJECTIONS = ["q_proj", "k_proj", "v_proj", "out_proj"]

def apply_linear_adapter(block, adapter_cls, r, alpha):
    attn = block.attn
    for name in ADAPTER_TARGET_PROJECTIONS:
        setattr(attn, name, adapter_cls(getattr(attn, name), r=r, alpha=alpha))


def apply_method(bb, cfg):
    m = cfg["method"]
    if m == "full":
        for p in bb.parameters(): p.requires_grad = True
        return bb
    for p in bb.parameters(): p.requires_grad = False
    if m == "lora":
        for blk in bb.blocks: apply_linear_adapter(blk, LoRALinear, cfg.get("r", 16), cfg.get("alpha", 32))
    elif m == "lorma":
        for blk in bb.blocks: apply_linear_adapter(blk, LoRMALinear, cfg.get("r", 16), cfg.get("alpha", 16))
    elif m in ("prefix", "prompt", "frozen"):
        pass   # these touch the *encoder*, not the backbone weights -- handled in SentEncoder below
    else:
        raise ValueError(m)
    return bb


def make_backbone(cfg):
    bb = copy.deepcopy(BASE_BACKBONE)
    if cfg.get("student_layers"):                 # KD student: keep only a subset of the Phase-1 blocks
        bb.blocks = nn.ModuleList([bb.blocks[i] for i in cfg["student_layers"]])
        bb.num_layers = len(bb.blocks)
    return apply_method(bb, cfg)


class PrefixTuningBank(nn.Module):
    """One learnable [prefix_len, d_model] 'virtual memory' per layer, fed into
    TransformerXLGQAModel's own `memory` argument."""
    def __init__(self, num_layers, prefix_len, d_model):
        super().__init__()
        self.prefix_len = prefix_len
        self.prefixes = nn.Parameter(torch.randn(num_layers, prefix_len, d_model) * 0.02)
    def get(self, batch_size, device):
        return [self.prefixes[i].unsqueeze(0).expand(batch_size, -1, -1).to(device) for i in range(self.prefixes.size(0))]


class PromptTuningEmbedding(nn.Module):
    def __init__(self, prompt_len, d_model):
        super().__init__()
        self.prompt_len = prompt_len
        self.prompt_embeds = nn.Parameter(torch.randn(prompt_len, d_model) * 0.02)
    def get(self, batch_size, device):
        return self.prompt_embeds.unsqueeze(0).expand(batch_size, -1, -1).to(device)

## 7. Sentence encoder (Transformer-XL + GQA) + document-level BiLSTM head

`encode_backbone` is a thin re-implementation of `TransformerXLGQAModel.forward` that stops **before**
`lm_head` and returns hidden states instead of logits -- this is the only way to get a feature vector per
token, since the original `forward` only returns next-token logits. It also knows how to splice in a
prefix bank (as `memory`) or a prompt bank (prepended to the token embeddings), so one function serves
all five adaptation types.

In [ ]:
def encode_backbone(backbone, input_ids, prefix_bank=None, prompt_bank=None):
    """input_ids: [B, T] (right-padded with PAD_ID). Returns hidden states [B, T', d_model]
    (T' = T, or T + prompt_len if a prompt bank is supplied) AFTER backbone.final_norm."""
    B = input_ids.size(0)
    device = input_ids.device
    x = backbone.embedding(input_ids)

    if prompt_bank is not None:
        prompt = prompt_bank.get(B, device)
        x = torch.cat([prompt, x], dim=1)

    Q = x.size(1)
    memory = None
    if prefix_bank is not None:
        memory = prefix_bank.get(B, device)
        mem_len = memory[0].size(1)
    else:
        mem_len = 0
    K = mem_len + Q
    rel_pos = backbone.pos_encoding(Q, K, device=device)
    for i, block in enumerate(backbone.blocks):
        mem_i = memory[i] if memory is not None else None
        x = block(x, mem_i, rel_pos)
    return backbone.final_norm(x)


class SentEncoder(nn.Module):
    """Transformer-XL + GQA -> one vector per sentence: concat(mean over tokens, last-token state).
    Sentences are sorted by length for efficient padding, same trick as the GQLA hierarchical notebook."""
    def __init__(self, backbone, prefix_bank=None, prompt_bank=None):
        super().__init__()
        self.bb = backbone
        self.prefix_bank = prefix_bank
        self.prompt_bank = prompt_bank

    def _run(self, id_lists):
        dev = self.bb.embedding.weight.device
        n = max(len(x) for x in id_lists); B = len(id_lists)
        ids = torch.full((B, n), PAD_ID, dtype=torch.long)
        mask = torch.zeros((B, n), dtype=torch.long)
        for i, x in enumerate(id_lists):
            ids[i, :len(x)] = torch.tensor(x); mask[i, :len(x)] = 1
        ids, mask = ids.to(dev), mask.to(dev)

        h = encode_backbone(self.bb, ids, prefix_bank=self.prefix_bank, prompt_bank=self.prompt_bank).float()

        if self.prompt_bank is not None:
            plen = self.prompt_bank.prompt_len
            h = h[:, plen:]                                   # drop the prompt positions, keep real tokens
        # (prefix tokens live in `memory`, not concatenated to h, so nothing to trim there)

        m = mask.unsqueeze(-1).float()
        mean = (h * m).sum(1) / m.sum(1).clamp_min(1)
        last = h[torch.arange(B, device=dev), (mask.sum(1) - 1).clamp_min(0)]
        return torch.cat([mean, last], -1)

    def forward(self, id_lists, micro=64):
        N = len(id_lists); order = sorted(range(N), key=lambda i: len(id_lists[i])); chunks = []
        for s in range(0, N, micro):
            chunks.append(self._run([id_lists[i] for i in order[s:s + micro]]))
        out = torch.cat(chunks, 0)
        inv = torch.empty(N, dtype=torch.long, device=out.device)
        inv[torch.tensor(order, device=out.device)] = torch.arange(N, device=out.device)
        return out[inv]


class RRHead(nn.Module):
    """sentence vector + position + domain -> 2-layer BiLSTM over the sentence sequence -> label logits.
    Identical architecture to the GQLA hierarchical notebook's head."""
    def __init__(self, in_dim, n_labels, proj=256, hid=192, use_context=True, use_pos=True, drop=0.3, feat_drop=0.2):
        super().__init__()
        self.use_context, self.use_pos = use_context, use_pos
        self.norm = nn.LayerNorm(in_dim); self.fdrop = nn.Dropout(feat_drop)
        self.proj = nn.Sequential(nn.Linear(in_dim, proj), nn.GELU())
        self.pos_lin = nn.Linear(N_POS_F, 32); self.pos_emb = nn.Embedding(N_POS_BINS, 32); self.dom_emb = nn.Embedding(2, 16)
        d_in = proj + (64 if use_pos else 0) + 16
        if use_context:
            self.lstm = nn.LSTM(d_in, hid, num_layers=2, batch_first=True, bidirectional=True, dropout=drop)
            d_out = 2 * hid + d_in
        else:
            d_out = d_in
        self.drop = nn.Dropout(drop)
        self.out = nn.Sequential(nn.Linear(d_out, 256), nn.GELU(), nn.Dropout(drop), nn.Linear(256, n_labels))

    def forward(self, feat, posf, posb, dom, lengths):
        x = self.proj(self.fdrop(self.norm(feat.float())))
        side = [self.dom_emb(dom)[:, None, :].expand(-1, x.size(1), -1)]
        if self.use_pos: side = [F.gelu(self.pos_lin(posf)), self.pos_emb(posb)] + side
        z = torch.cat([x] + side, -1)
        if self.use_context:
            pk = nn.utils.rnn.pack_padded_sequence(z, lengths.cpu(), batch_first=True, enforce_sorted=False)
            o, _ = self.lstm(pk); o, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True, total_length=z.size(1))
            z = torch.cat([o, z], -1)
        return self.out(self.drop(z))


class RRModel(nn.Module):
    def __init__(self, encoder, head):
        super().__init__(); self.encoder = encoder; self.head = head


def make_model(cfg, frozen_features=False, use_context=True, use_pos=True):
    if frozen_features:
        return RRModel(None, RRHead(FEAT_DIM, K, use_context=use_context, use_pos=use_pos)).to(DEVICE)
    backbone = make_backbone(cfg)
    prefix_bank = prompt_bank = None
    if cfg["method"] == "prefix":
        prefix_bank = PrefixTuningBank(backbone.num_layers, cfg.get("prefix_len", PREFIX_LEN), D_MODEL).to(DEVICE)
    elif cfg["method"] == "prompt":
        prompt_bank = PromptTuningEmbedding(cfg.get("prompt_len", PROMPT_LEN), D_MODEL).to(DEVICE)
    enc = SentEncoder(backbone, prefix_bank=prefix_bank, prompt_bank=prompt_bank).to(DEVICE)
    model = RRModel(enc, RRHead(FEAT_DIM, K, use_context=use_context, use_pos=use_pos)).to(DEVICE)
    return model


def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    return tr, tot

## 8. Windowed training loop, prediction, experiment runner (with resume)

In [ ]:
def make_windows(docs, rng, shuffle=True):
    items = []
    for d in docs:
        n = d["n"]; off = int(rng.integers(0, WINDOW)) if shuffle else 0
        cuts = sorted(set([0] + list(range(off if off > 0 else WINDOW, n, WINDOW)) + [n]))
        for s, e in zip(cuts[:-1], cuts[1:]):
            if e - s >= 4 or n < 4 or not shuffle: items.append((d, s, e))
    if shuffle: rng.shuffle(items)
    return items

def build_batch(items, use_soft=False):
    B = len(items); L = max(e - s for _, s, e in items)
    posf = np.zeros((B, L, N_POS_F), np.float32); posb = np.zeros((B, L), np.int64)
    y = np.full((B, L), -100, np.int64); soft = np.zeros((B, L, K), np.float32) if use_soft else None
    ids, bi, li, lens, dom, feat = [], [], [], [], [], None
    if "feat" in items[0][0]: feat = np.zeros((B, L, FEAT_DIM), np.float32)
    for b, (d, s, e) in enumerate(items):
        m = e - s; posf[b, :m] = d["posf"][s:e]; posb[b, :m] = d["posb"][s:e]; lens.append(m); dom.append(d["domain"])
        if d["y"] is not None: y[b, :m] = d["y"][s:e]
        if use_soft: soft[b, :m] = d["soft"][s:e]
        if feat is not None: feat[b, :m] = d["feat"][s:e]
        ids += d["ids"][s:e]; bi += [b] * m; li += list(range(m))
    t = lambda a: torch.from_numpy(a).to(DEVICE)
    return dict(ids=ids, bi=torch.tensor(bi, device=DEVICE), li=torch.tensor(li, device=DEVICE), B=B, L=L,
                posf=t(posf), posb=t(posb), y=t(y), soft=t(soft) if use_soft else None, dom=torch.tensor(dom, device=DEVICE),
                lens=torch.tensor(lens), feat=t(feat) if feat is not None else None)

def forward_batch(model, bt, micro):
    if model.encoder is not None:
        flat = model.encoder(bt["ids"], micro=micro)
        feats = flat.new_zeros(bt["B"], bt["L"], flat.size(-1)); feats[bt["bi"], bt["li"]] = flat
    else:
        feats = bt["feat"]
    return model.head(feats, bt["posf"], bt["posb"], bt["dom"], bt["lens"])

def rr_loss(logits, bt, cfg):
    valid = bt["y"] != -100; lg = logits[valid].float(); yy = bt["y"][valid]
    ce = F.cross_entropy(lg, yy, weight=CLASS_W, label_smoothing=LABEL_SMOOTH)
    if bt["soft"] is None: return ce
    T, a = cfg["kd_T"], cfg["kd_alpha"]
    kl = F.kl_div(F.log_softmax(lg / T, -1), bt["soft"][valid], reduction="batchmean") * T * T
    return a * kl + (1 - a) * ce

@torch.no_grad()
def predict_docs(model, docs):
    model.eval(); out = {}
    for d in docs:
        bt = build_batch([(d, 0, d["n"])])
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            lg = forward_batch(model, bt, ENC_MICRO_EVAL)
        out[d["id"]] = F.softmax(lg[0].float(), -1).cpu().numpy()
    return out

def predict_all(model):
    return {s: predict_docs(model, DOCS[s]) for s in SPLITS}

def train_model(model, cfg, train_docs, use_soft=False, verbose=True, seed=SEED):
    set_seed(seed); rng = np.random.default_rng(seed)
    bb_params = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("head.")]
    groups = [dict(params=list(model.head.parameters()), lr=HEAD_LR, weight_decay=HEAD_WD)]
    if bb_params: groups.append(dict(params=bb_params, lr=cfg["lr"], weight_decay=0.01 if cfg["method"] == "full" else 0.0))
    opt = torch.optim.AdamW(groups); base_lrs = [g["lr"] for g in groups]
    steps_ep = math.ceil(len(make_windows(train_docs, np.random.default_rng(0))) / WINDOWS_PER_BATCH)
    total = steps_ep * cfg["epochs"]; warm = max(1, int(0.06 * total)); step = 0
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    clip_params = [p for p in model.parameters() if p.requires_grad]
    best, best_state, best_ep, bad, hist = -1, None, 0, 0, []
    peak0 = time.time()
    if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
    for ep in range(1, cfg["epochs"] + 1):
        model.train(); items = make_windows(train_docs, rng); tl, nb = 0., 0
        for i in range(0, len(items), WINDOWS_PER_BATCH):
            bt = build_batch(items[i:i + WINDOWS_PER_BATCH], use_soft)
            f = step / warm if step < warm else 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm))) * 0.9 + 0.1
            for g, b in zip(opt.param_groups, base_lrs): g["lr"] = b * f
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = forward_batch(model, bt, ENC_MICRO_TRAIN)
            loss = rr_loss(logits, bt, cfg)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(clip_params, GRAD_CLIP); scaler.step(opt); scaler.update()
            tl += loss.item(); nb += 1; step += 1
        dp = {s: predict_docs(model, DOCS[s]) for s in ("CL_dev", "IT_dev")}; r = eval_dev(dp)
        hist.append(dict(epoch=ep, train_loss=tl / nb, **{k: round(v, 4) for k, v in r.items()}))
        if verbose: print(f"ep{ep:2d} loss {tl/nb:.3f} | devF1 CL {r['CL']:.3f} IT {r['IT']:.3f} mean {r['mean']:.3f} (13-label {r['mean_13']:.3f})")
        if r["mean"] > best:
            best, best_ep, bad = r["mean"], ep, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}
        else:
            bad += 1
            if bad >= PATIENCE: print("early stop"); break
    with torch.no_grad():
        for k, v in model.named_parameters():
            if k in best_state: v.copy_(best_state[k].to(v.device))
    peak = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0.0
    return dict(best_epoch=best_ep, history=hist, train_sec=time.time() - peak0, peak_gb=peak), best_state

RESULTS = {}
def run_experiment(name, cfg, use_soft=False, frozen_features=False, use_context=True, use_pos=True, seed=SEED):
    path = os.path.join(PROB_DIR, name + ".pkl")
    if os.path.exists(path) and not FORCE_RERUN:
        RESULTS[name] = pickle.load(open(path, "rb")); r = RESULTS[name]
        print(f"[{name}] loaded from disk: dev mean F1 {r['dev']['mean']:.4f}"); return r
    print(f"=== {name}: {cfg}")
    model = make_model(cfg, frozen_features, use_context, use_pos); tr, tot = count_params(model)
    print(f"trainable {tr/1e6:.2f}M / total {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")
    info, best_state = train_model(model, cfg, TRAIN_DOCS, use_soft, seed=seed)
    t0 = time.time(); probs = predict_all(model); inf = time.time() - t0
    n_dev_test = sum(DOCS[s][i]["n"] for s in SPLITS for i in range(len(DOCS[s])))
    res = dict(name=name, cfg=cfg, trainable=tr, total=tot, probs=probs, dev=eval_dev(probs),
               sent_per_sec=n_dev_test / inf, **info)
    pickle.dump(res, open(path, "wb")); torch.save(best_state, os.path.join(OUTPUT_DIR, f"ckpt_{name}.pt"))
    RESULTS[name] = res
    print(f"[{name}] best epoch {info['best_epoch']} | dev F1 CL {res['dev']['CL']:.3f} IT {res['dev']['IT']:.3f} mean {res['dev']['mean']:.3f} "
          f"| {info['train_sec']/60:.1f} min | peak {info['peak_gb']:.1f} GB")
    del model, best_state; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return res

## 9. Cheap baselines first (minutes): frozen Phase-1 features with / without document context
This answers *"does context help?"*: compare `A0` (sentence only) → `A1` (+position) → `E0` (+BiLSTM
context), all built on the **same frozen Transformer-XL + GQA features**. If `E0` clearly beats `A0`/`A1`,
the BiLSTM-over-sentences idea is doing real work on this backbone before you spend time on the
fine-tuning experiments below.

In [ ]:
def cache_features():
    enc = SentEncoder(make_backbone(dict(method="frozen"))).to(DEVICE).eval()
    with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
        for s in SPLITS:
            for d in DOCS[s]: d["feat"] = enc(d["ids"], micro=ENC_MICRO_EVAL).float().cpu().numpy()
    del enc; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
cache_features(); print("cached frozen features:", FEAT_DIM, "dims per sentence")

FROZEN = dict(method="frozen", lr=0.0, epochs=30)
run_experiment("A0_frozen_sentence_only", FROZEN, frozen_features=True, use_context=False, use_pos=False)
run_experiment("A1_frozen_sentence_plus_position", FROZEN, frozen_features=True, use_context=False, use_pos=True)
seed_runs = [run_experiment(f"A2_frozen_bilstm_s{s}", FROZEN, frozen_features=True, use_context=True, use_pos=True, seed=s) for s in (0, 1, 2)]

avg = {sp: {d["id"]: np.mean([r["probs"][sp][d["id"]] for r in seed_runs], 0) for d in DOCS[sp]} for sp in SPLITS}
RESULTS["E0_frozen_bilstm"] = dict(name="E0_frozen_bilstm", cfg=FROZEN, probs=avg, dev=eval_dev(avg), trainable=seed_runs[0]["trainable"],
                                   total=seed_runs[0]["total"], best_epoch=int(np.mean([r["best_epoch"] for r in seed_runs])),
                                   train_sec=sum(r["train_sec"] for r in seed_runs), peak_gb=0.0, sent_per_sec=float("nan"))
for k in ("A0_frozen_sentence_only", "A1_frozen_sentence_plus_position", "E0_frozen_bilstm"):
    d = RESULTS[k]["dev"]; print(f"{k:36s} CL {d['CL']:.3f}  IT {d['IT']:.3f}  mean {d['mean']:.3f}  (strict-13 {d['mean_13']:.3f})")

cached frozen features: 1536 dims per sentence
=== A0_frozen_sentence_only: {'method': 'frozen', 'lr': 0.0, 'epochs': 30}
trainable 0.47M / total 0.47M (100.00%)
ep 1 loss 2.577 | devF1 CL 0.034 IT 0.038 mean 0.036 (13-label 0.029)
ep 2 loss 2.562 | devF1 CL 0.030 IT 0.024 mean 0.027 (13-label 0.022)
ep 3 loss 2.512 | devF1 CL 0.030 IT 0.024 mean 0.027 (13-label 0.022)
ep 4 loss 2.492 | devF1 CL 0.030 IT 0.038 mean 0.034 (13-label 0.028)
ep 5 loss 2.491 | devF1 CL 0.030 IT 0.024 mean 0.027 (13-label 0.022)
ep 6 loss 2.492 | devF1 CL 0.030 IT 0.014 mean 0.022 (13-label 0.018)
early stop
[A0_frozen_sentence_only] best epoch 1 | dev F1 CL 0.034 IT 0.038 mean 0.036 | 0.2 min | peak 0.0 GB
=== A1_frozen_sentence_plus_position: {'method': 'frozen', 'lr': 0.0, 'epochs': 30}
trainable 0.49M / total 0.49M (100.00%)
ep 1 loss 2.500 | devF1 CL 0.133 IT 0.129 mean 0.131 (13-label 0.106)
ep 2 loss 2.306 | devF1 CL 0.091 IT 0.141 mean 0.116 (13-label 0.094)
ep 3 loss 2.235 | devF1 CL 0.153 IT 0.184 

## 10. The six required experiments (end-to-end, Transformer-XL + GQA + document-level BiLSTM head)

In [ ]:
# drop cached features so the end-to-end runs really use the encoder
for s in SPLITS:
    for d in DOCS[s]: d.pop("feat", None)
run_experiment("full_finetune", EXPERIMENTS["full_finetune"])

=== full_finetune: {'method': 'full', 'lr': 3e-05, 'epochs': 10}
trainable 78.25M / total 78.25M (100.00%)
ep 1 loss 2.449 | devF1 CL 0.131 IT 0.161 mean 0.146 (13-label 0.118)
ep 2 loss 2.247 | devF1 CL 0.091 IT 0.116 mean 0.103 (13-label 0.084)
ep 3 loss 2.188 | devF1 CL 0.202 IT 0.150 mean 0.176 (13-label 0.141)
ep 4 loss 2.131 | devF1 CL 0.248 IT 0.162 mean 0.205 (13-label 0.164)
ep 5 loss 2.115 | devF1 CL 0.196 IT 0.182 mean 0.189 (13-label 0.152)
ep 6 loss 2.092 | devF1 CL 0.213 IT 0.193 mean 0.203 (13-label 0.164)
ep 7 loss 2.059 | devF1 CL 0.215 IT 0.203 mean 0.209 (13-label 0.169)
ep 8 loss 2.034 | devF1 CL 0.238 IT 0.203 mean 0.221 (13-label 0.178)
ep 9 loss 2.018 | devF1 CL 0.231 IT 0.188 mean 0.209 (13-label 0.168)
ep10 loss 2.009 | devF1 CL 0.236 IT 0.222 mean 0.229 (13-label 0.184)
[full_finetune] best epoch 10 | dev F1 CL 0.236 IT 0.222 mean 0.229 | 12.3 min | peak 4.2 GB


{'name': 'full_finetune',
 'cfg': {'method': 'full', 'lr': 3e-05, 'epochs': 10},
 'trainable': 78247053,
 'total': 78247053,
 'probs': {'CL_train': {'CCI_All_India_Distillers_Association_vs_Haldyn_Glass_GCO100028COM855733': array([[7.73110986e-01, 6.16285801e-02, 1.47699974e-02, 4.93918266e-03,
           3.69025674e-03, 1.01593193e-02, 3.89330206e-04, 3.41626420e-03,
           6.11489788e-02, 2.51691211e-02, 1.23469028e-04, 6.71489956e-03,
           3.47397178e-02],
          [7.33559966e-01, 1.01333506e-01, 3.60786878e-02, 7.87144247e-03,
           1.48265595e-02, 5.48710907e-03, 6.71538175e-04, 4.96203359e-03,
           5.57979681e-02, 3.40586863e-02, 1.59193078e-04, 3.03619378e-03,
           2.15719291e-03],
          [8.68616104e-01, 1.21690510e-02, 2.80994214e-02, 6.22711424e-03,
           2.78629782e-03, 2.77814665e-03, 5.51878766e-04, 3.60221788e-03,
           4.70806584e-02, 1.90967545e-02, 2.92528712e-04, 2.77814665e-03,
           5.92167862e-03],
          [8.3754742

In [ ]:
run_experiment("lora", EXPERIMENTS["lora"])

=== lora: {'method': 'lora', 'lr': 0.0003, 'epochs': 12, 'r': 16, 'alpha': 32}
trainable 2.93M / total 78.89M (3.71%)
ep 1 loss 2.474 | devF1 CL 0.128 IT 0.153 mean 0.140 (13-label 0.114)
ep 2 loss 2.285 | devF1 CL 0.101 IT 0.138 mean 0.120 (13-label 0.097)
ep 3 loss 2.204 | devF1 CL 0.207 IT 0.159 mean 0.183 (13-label 0.147)
ep 4 loss 2.139 | devF1 CL 0.208 IT 0.184 mean 0.196 (13-label 0.158)
ep 5 loss 2.126 | devF1 CL 0.190 IT 0.183 mean 0.186 (13-label 0.150)
ep 6 loss 2.098 | devF1 CL 0.207 IT 0.214 mean 0.210 (13-label 0.170)
ep 7 loss 2.072 | devF1 CL 0.216 IT 0.177 mean 0.197 (13-label 0.158)
ep 8 loss 2.032 | devF1 CL 0.227 IT 0.216 mean 0.221 (13-label 0.178)
ep 9 loss 2.023 | devF1 CL 0.226 IT 0.201 mean 0.214 (13-label 0.172)
ep10 loss 2.004 | devF1 CL 0.239 IT 0.228 mean 0.233 (13-label 0.188)
ep11 loss 1.996 | devF1 CL 0.235 IT 0.204 mean 0.220 (13-label 0.177)
ep12 loss 1.971 | devF1 CL 0.233 IT 0.199 mean 0.216 (13-label 0.174)
[lora] best epoch 10 | dev F1 CL 0.239 IT 

{'name': 'lora',
 'cfg': {'method': 'lora', 'lr': 0.0003, 'epochs': 12, 'r': 16, 'alpha': 32},
 'trainable': 2926221,
 'total': 78886029,
 'probs': {'CL_train': {'CCI_All_India_Distillers_Association_vs_Haldyn_Glass_GCO100028COM855733': array([[7.82074332e-01, 6.00133426e-02, 2.39416994e-02, 5.04794391e-03,
           3.02015804e-03, 9.02753882e-03, 2.28831588e-04, 1.75992295e-03,
           5.48566431e-02, 2.79087964e-02, 1.27363950e-04, 7.67467823e-03,
           2.43187249e-02],
          [7.28873014e-01, 8.62897187e-02, 5.09754010e-02, 8.99769366e-03,
           1.14437845e-02, 7.33295409e-03, 5.06624696e-04, 3.55812581e-03,
           5.24909161e-02, 4.22394648e-02, 2.15357984e-04, 4.35950654e-03,
           2.71747354e-03],
          [8.38094532e-01, 1.31183388e-02, 3.91599424e-02, 8.61070305e-03,
           1.81728159e-03, 3.96156311e-03, 3.77589662e-04, 3.01969028e-03,
           5.30051328e-02, 2.58074235e-02, 2.87255098e-04, 4.82545188e-03,
           7.91514013e-03],
       

In [ ]:
run_experiment("lorma", EXPERIMENTS["lorma"])  # LORMA fine-tuning is compulsory per the capstone document (Section 9.2.1).

=== lorma: {'method': 'lorma', 'lr': 0.0003, 'epochs': 12, 'r': 16, 'alpha': 16}
trainable 2.78M / total 78.74M (3.53%)
ep 1 loss 2.430 | devF1 CL 0.162 IT 0.153 mean 0.157 (13-label 0.127)
ep 2 loss 2.254 | devF1 CL 0.090 IT 0.140 mean 0.115 (13-label 0.094)
ep 3 loss 2.198 | devF1 CL 0.200 IT 0.193 mean 0.197 (13-label 0.159)
ep 4 loss 2.140 | devF1 CL 0.185 IT 0.222 mean 0.203 (13-label 0.165)
ep 5 loss 2.127 | devF1 CL 0.199 IT 0.171 mean 0.185 (13-label 0.149)
ep 6 loss 2.102 | devF1 CL 0.203 IT 0.217 mean 0.210 (13-label 0.170)
ep 7 loss 2.076 | devF1 CL 0.224 IT 0.203 mean 0.213 (13-label 0.172)
ep 8 loss 2.043 | devF1 CL 0.239 IT 0.229 mean 0.234 (13-label 0.189)
ep 9 loss 2.026 | devF1 CL 0.257 IT 0.216 mean 0.237 (13-label 0.190)
ep10 loss 2.008 | devF1 CL 0.243 IT 0.248 mean 0.246 (13-label 0.199)
ep11 loss 2.003 | devF1 CL 0.251 IT 0.215 mean 0.233 (13-label 0.188)
ep12 loss 1.983 | devF1 CL 0.227 IT 0.225 mean 0.226 (13-label 0.183)
[lorma] best epoch 10 | dev F1 CL 0.243 

{'name': 'lorma',
 'cfg': {'method': 'lorma', 'lr': 0.0003, 'epochs': 12, 'r': 16, 'alpha': 16},
 'trainable': 2778765,
 'total': 78738573,
 'probs': {'CL_train': {'CCI_All_India_Distillers_Association_vs_Haldyn_Glass_GCO100028COM855733': array([[7.80875802e-01, 8.07906613e-02, 1.98857989e-02, 4.36193636e-03,
           3.91383469e-03, 6.37764577e-03, 3.08055751e-04, 1.84335629e-03,
           5.91654591e-02, 1.95488427e-02, 1.69792591e-04, 8.12136382e-03,
           1.46373920e-02],
          [7.23131895e-01, 9.72935110e-02, 5.19251190e-02, 7.22563174e-03,
           1.74976159e-02, 4.51947749e-03, 5.84205787e-04, 4.09498671e-03,
           5.98238334e-02, 2.73826607e-02, 2.65906274e-04, 4.95398929e-03,
           1.30118988e-03],
          [8.59590888e-01, 1.18783601e-02, 3.49977799e-02, 6.64366363e-03,
           2.41677300e-03, 2.52041942e-03, 4.03489074e-04, 3.71041591e-03,
           5.29499501e-02, 1.64633114e-02, 3.10576608e-04, 3.01066088e-03,
           5.10383211e-03],
     

In [ ]:
run_experiment("prefix_tuning", EXPERIMENTS["prefix_tuning"])

=== prefix_tuning: {'method': 'prefix', 'lr': 0.0005, 'epochs': 12, 'prefix_len': 32}
trainable 2.48M / total 78.44M (3.17%)
ep 1 loss 2.446 | devF1 CL 0.129 IT 0.164 mean 0.146 (13-label 0.119)
ep 2 loss 2.263 | devF1 CL 0.091 IT 0.145 mean 0.118 (13-label 0.097)
ep 3 loss 2.201 | devF1 CL 0.208 IT 0.172 mean 0.190 (13-label 0.153)
ep 4 loss 2.137 | devF1 CL 0.204 IT 0.203 mean 0.203 (13-label 0.164)
ep 5 loss 2.123 | devF1 CL 0.189 IT 0.184 mean 0.186 (13-label 0.150)
ep 6 loss 2.101 | devF1 CL 0.205 IT 0.202 mean 0.204 (13-label 0.165)
ep 7 loss 2.063 | devF1 CL 0.231 IT 0.187 mean 0.209 (13-label 0.168)
ep 8 loss 2.032 | devF1 CL 0.238 IT 0.233 mean 0.235 (13-label 0.190)
ep 9 loss 2.007 | devF1 CL 0.233 IT 0.199 mean 0.216 (13-label 0.174)
ep10 loss 1.999 | devF1 CL 0.250 IT 0.237 mean 0.244 (13-label 0.197)
ep11 loss 1.996 | devF1 CL 0.241 IT 0.225 mean 0.233 (13-label 0.188)
ep12 loss 1.973 | devF1 CL 0.225 IT 0.218 mean 0.221 (13-label 0.179)
[prefix_tuning] best epoch 10 | dev

{'name': 'prefix_tuning',
 'cfg': {'method': 'prefix', 'lr': 0.0005, 'epochs': 12, 'prefix_len': 32},
 'trainable': 2483853,
 'total': 78443661,
 'probs': {'CL_train': {'CCI_All_India_Distillers_Association_vs_Haldyn_Glass_GCO100028COM855733': array([[8.17863166e-01, 6.08881637e-02, 2.21061353e-02, 2.50335666e-03,
           4.45835991e-03, 6.02874439e-03, 2.94923491e-04, 2.06726464e-03,
           4.87818867e-02, 1.64660309e-02, 1.61920674e-04, 4.33386909e-03,
           1.40461549e-02],
          [7.38150656e-01, 9.36621949e-02, 6.02959618e-02, 3.64233344e-03,
           1.68335009e-02, 4.22930857e-03, 5.88241499e-04, 3.61046102e-03,
           4.79315035e-02, 2.56058238e-02, 1.93225205e-04, 2.96408893e-03,
           2.29270849e-03],
          [8.74575019e-01, 1.16907349e-02, 3.48509997e-02, 2.74310284e-03,
           3.06015229e-03, 3.16346460e-03, 3.11091513e-04, 3.25433561e-03,
           4.35000286e-02, 1.47676552e-02, 1.98515729e-04, 2.73240847e-03,
           5.15239034e-03],


In [ ]:
run_experiment("prompt_tuning", EXPERIMENTS["prompt_tuning"])

=== prompt_tuning: {'method': 'prompt', 'lr': 0.0005, 'epochs': 12, 'prompt_len': 32}
trainable 2.31M / total 78.27M (2.95%)
ep 1 loss 2.464 | devF1 CL 0.129 IT 0.137 mean 0.133 (13-label 0.108)
ep 2 loss 2.274 | devF1 CL 0.093 IT 0.148 mean 0.120 (13-label 0.098)
ep 3 loss 2.205 | devF1 CL 0.204 IT 0.175 mean 0.190 (13-label 0.153)
ep 4 loss 2.135 | devF1 CL 0.266 IT 0.199 mean 0.233 (13-label 0.187)
ep 5 loss 2.119 | devF1 CL 0.202 IT 0.170 mean 0.186 (13-label 0.149)
ep 6 loss 2.096 | devF1 CL 0.215 IT 0.202 mean 0.209 (13-label 0.168)
ep 7 loss 2.067 | devF1 CL 0.235 IT 0.181 mean 0.208 (13-label 0.167)
ep 8 loss 2.040 | devF1 CL 0.232 IT 0.233 mean 0.232 (13-label 0.188)
ep 9 loss 2.017 | devF1 CL 0.248 IT 0.188 mean 0.218 (13-label 0.175)
early stop
[prompt_tuning] best epoch 4 | dev F1 CL 0.266 IT 0.199 mean 0.233 | 13.0 min | peak 4.1 GB


{'name': 'prompt_tuning',
 'cfg': {'method': 'prompt', 'lr': 0.0005, 'epochs': 12, 'prompt_len': 32},
 'trainable': 2311821,
 'total': 78271629,
 'probs': {'CL_train': {'CCI_All_India_Distillers_Association_vs_Haldyn_Glass_GCO100028COM855733': array([[8.39257360e-01, 2.76851524e-02, 2.08189189e-02, 8.09232239e-03,
           4.03346913e-03, 5.66592719e-03, 5.76554332e-04, 5.81730856e-03,
           5.07470407e-02, 1.70277245e-02, 6.12541393e-04, 2.65816133e-03,
           1.70074683e-02],
          [7.90280342e-01, 4.63379882e-02, 4.42808308e-02, 1.01048220e-02,
           9.29537229e-03, 4.40155575e-03, 7.82249961e-04, 6.44186093e-03,
           5.55948690e-02, 2.42861155e-02, 8.44163471e-04, 2.37678131e-03,
           4.97302553e-03],
          [8.73069406e-01, 9.66583099e-03, 2.50467863e-02, 9.50668752e-03,
           2.71707028e-03, 3.89202312e-03, 8.39246670e-04, 6.39496604e-03,
           4.54203524e-02, 1.33657278e-02, 1.20687054e-03, 1.85831706e-03,
           7.01664109e-03],


### Knowledge distillation (mandatory 6th experiment)
**Teacher:** the fully fine-tuned `full_finetune` model (backbone + BiLSTM head), frozen.

**Student backbone:** a `num_layers / STUDENT_LAYER_STRIDE`-layer Transformer-XL + GQA model, warm-started
by copying every `STUDENT_LAYER_STRIDE`-th block (plus embedding and final norm) from the teacher's
Phase-1-derived weights -- the same DistilBERT-style layer-selection initialization used in the original
Transformer-XL notebook. The student gets its **own** BiLSTM head, trained with hard-label cross-entropy
+ soft-label KL-divergence against the teacher's per-sentence probability distribution (temperature-scaled).

In [ ]:
def build_student_backbone(teacher_backbone, stride=STUDENT_LAYER_STRIDE):
    teacher_layers = teacher_backbone.num_layers
    student_layers = max(2, teacher_layers // stride)
    keep_idx = torch.linspace(0, teacher_layers - 1, steps=student_layers).round().long().tolist()
    student = TransformerXLGQAModel(
        vocab_size=MODEL_CONFIG["vocab_size"], d_model=MODEL_CONFIG["d_model"], num_layers=student_layers,
        num_query_heads=MODEL_CONFIG["num_query_heads"], num_kv_heads=MODEL_CONFIG["num_kv_heads"],
        memory_length=MODEL_CONFIG["memory_length"], dropout=MODEL_CONFIG["dropout"],
    )
    del student.lm_head
    student.embedding.load_state_dict(teacher_backbone.embedding.state_dict())
    student.final_norm.load_state_dict(teacher_backbone.final_norm.state_dict())
    for s_i, t_i in enumerate(keep_idx):
        student.blocks[s_i].load_state_dict(teacher_backbone.blocks[t_i].state_dict())
    print(f"Student backbone: {student_layers} layers (teacher had {teacher_layers}), copied from teacher layers {keep_idx}")
    return student.to(DEVICE)


def attach_soft_targets(teacher_model, T):
    teacher_model.eval()
    with torch.no_grad():
        for split in ("CL_train", "IT_train"):
            for d in DOCS[split]:
                bt = build_batch([(d, 0, d["n"])])
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                    lg = forward_batch(teacher_model, bt, ENC_MICRO_EVAL)[0].float()
                p = F.softmax(lg, -1).cpu().numpy()
                q = np.exp(np.log(p + 1e-6) / T)
                d["soft"] = (q / q.sum(1, keepdims=True)).astype(np.float32)

exp_name = "knowledge_distillation"
path = os.path.join(PROB_DIR, exp_name + ".pkl")
if os.path.exists(path) and not FORCE_RERUN:
    RESULTS[exp_name] = pickle.load(open(path, "rb"))
    print(f"[{exp_name}] loaded from disk: dev mean F1 {RESULTS[exp_name]['dev']['mean']:.4f}")
else:
    teacher = RESULTS["full_finetune"]
    teacher_model = make_model(EXPERIMENTS["full_finetune"])
    teacher_model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "ckpt_full_finetune.pt"), map_location=DEVICE), strict=False)
    for p in teacher_model.parameters(): p.requires_grad = False
    teacher_model.eval()

    attach_soft_targets(teacher_model, EXPERIMENTS["knowledge_distillation"]["kd_T"])

    cfg = EXPERIMENTS["knowledge_distillation"]
    student_backbone = build_student_backbone(teacher_model.encoder.bb, cfg["student_stride"])
    student_enc = SentEncoder(student_backbone).to(DEVICE)
    student = RRModel(student_enc, RRHead(FEAT_DIM, K)).to(DEVICE)
    for p in student.parameters(): p.requires_grad = True

    tr, tot = count_params(student); teacher_total = sum(p.numel() for p in teacher_model.parameters())
    print(f"Student trainable/total params: {tr/1e6:.2f}M / {tot/1e6:.2f}M  ({100*tot/teacher_total:.1f}% of teacher size)")

    info, best_state = train_model(student, cfg, TRAIN_DOCS, use_soft=True)
    for split in ("CL_train", "IT_train"):
        for d in DOCS[split]: d.pop("soft", None)

    probs = predict_all(student)
    res = dict(name=exp_name, cfg=cfg, trainable=tr, total=tot, probs=probs, dev=eval_dev(probs),
               sent_per_sec=float("nan"), **info)
    pickle.dump(res, open(path, "wb")); torch.save(best_state, os.path.join(OUTPUT_DIR, f"ckpt_{exp_name}.pt"))
    RESULTS[exp_name] = res
    print(f"[{exp_name}] best epoch {info['best_epoch']} | dev F1 CL {res['dev']['CL']:.3f} IT {res['dev']['IT']:.3f} mean {res['dev']['mean']:.3f}")
    del teacher_model, student, best_state; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

Student backbone: 2 layers (teacher had 8), copied from teacher layers [0, 7]
Student trainable/total params: 40.15M / 40.15M  (51.3% of teacher size)
ep 1 loss 0.960 | devF1 CL 0.159 IT 0.149 mean 0.154 (13-label 0.124)
ep 2 loss 0.723 | devF1 CL 0.187 IT 0.173 mean 0.180 (13-label 0.145)
ep 3 loss 0.689 | devF1 CL 0.211 IT 0.191 mean 0.201 (13-label 0.162)
ep 4 loss 0.662 | devF1 CL 0.224 IT 0.250 mean 0.237 (13-label 0.192)
ep 5 loss 0.657 | devF1 CL 0.192 IT 0.187 mean 0.189 (13-label 0.153)
ep 6 loss 0.648 | devF1 CL 0.213 IT 0.216 mean 0.214 (13-label 0.173)
ep 7 loss 0.636 | devF1 CL 0.235 IT 0.205 mean 0.220 (13-label 0.177)
ep 8 loss 0.630 | devF1 CL 0.245 IT 0.212 mean 0.228 (13-label 0.184)
ep 9 loss 0.628 | devF1 CL 0.237 IT 0.188 mean 0.212 (13-label 0.170)
early stop
[knowledge_distillation] best epoch 4 | dev F1 CL 0.224 IT 0.250 mean 0.237


## 11. Results table

In [ ]:
import pandas as pd
rows = []
for n, r in RESULTS.items():
    d = r["dev"]
    rows.append(dict(experiment=n, method=r["cfg"].get("method"), trainable_M=round(r["trainable"] / 1e6, 3),
                     total_M=round(r["total"] / 1e6, 2), best_epoch=r["best_epoch"], train_min=round(r["train_sec"] / 60, 1),
                     peak_GB=round(r["peak_gb"], 1), sent_per_s=None if np.isnan(r["sent_per_sec"]) else int(r["sent_per_sec"]),
                     CL_F1=round(d["CL"], 4), IT_F1=round(d["IT"], 4), mean_F1=round(d["mean"], 4),
                     pooled_F1=round(d["pooled"], 4), mean_F1_strict13=round(d["mean_13"], 4)))
SUMMARY = pd.DataFrame(rows).sort_values("mean_F1", ascending=False); display(SUMMARY)
SUMMARY.to_csv(os.path.join(OUTPUT_DIR, "results_summary.csv"), index=False)

,experiment,method,trainable_M,total_M,best_epoch,train_min,peak_GB,sent_per_s,CL_F1,IT_F1,mean_F1,pooled_F1,mean_F1_strict13
1,A1_frozen_sentence_plus_position,frozen,0.487,0.49,10,0.4,0.0,77699.0,0.2830,0.2207,0.2518,0.2647,0.2022
4,A2_frozen_bilstm_s2,frozen,2.287,2.29,7,1.0,0.1,15724.0,0.2969,0.1993,0.2481,0.2661,0.1985
5,E0_frozen_bilstm,frozen,2.287,2.29,6,2.7,0.0,NaN,0.2767,0.2180,0.2474,0.2766,0.1987
8,lorma,lorma,2.779,78.74,10,12.2,3.3,850.0,0.2432,0.2481,0.2456,0.2673,0.1985
9,prefix_tuning,prefix,2.484,78.44,10,13.2,3.3,708.0,0.2504,0.2369,0.2436,0.2593,0.1965
11,knowledge_distillation,full,40.153,40.15,4,3.7,1.7,NaN,0.2244,0.2505,0.2374,0.2524,0.1923
2,A2_frozen_bilstm_s0,frozen,2.287,2.29,9,1.1,0.1,15770.0,0.2675,0.2010,0.2343,0.2498,0.1879
7,lora,lora,2.926,78.89,10,12.5,3.2,804.0,0.2390,0.2276,0.2333,0.2488,0.1882
10,prompt_tuning,prompt,2.312,78.27,4,13.0,4.1,518.0,0.2665,0.1995,0.2330,0.2065,0.1869
6,full_finetune,full,78.247,78.25,10,12.3,4.2,883.0,0.2355,0.2216,0.2285,0.2475,0.1843


## 12. Ensemble + decoding (prior correction, Viterbi transition smoothing)
Dev has only 5 documents per domain, so choices are made conservatively: an ensemble of everything is
kept unless a smaller subset is clearly (>0.01) better, and decoding settings must beat plain argmax by
>0.003 to be adopted.

In [ ]:
def viterbi(logp, logT):
    n = len(logp); dp = logp[0].copy(); bp = np.zeros((n, K), dtype=np.int64)
    for t in range(1, n):
        sc = dp[:, None] + logT; bp[t] = sc.argmax(0); dp = sc.max(0) + logp[t]
    path = [int(dp.argmax())]
    for t in range(n - 1, 0, -1): path.append(int(bp[t][path[-1]]))
    return np.array(path[::-1])

def make_decoder(tau, lam):
    def dec(p, dom):
        lp = np.log(p + 1e-8) - tau * np.log(PRIOR)
        return lp.argmax(1) if lam <= 0 else viterbi(lp, lam * TRANS[dom])
    return dec

def ens_probs(names, weights=None):
    w = np.ones(len(names)) if weights is None else np.asarray(weights, float); w = w / w.sum()
    return {s: {d["id"]: sum(wi * RESULTS[n]["probs"][s][d["id"]] for wi, n in zip(w, names)) for d in DOCS[s]} for s in SPLITS}

MEMBERS = [n for n in ("E0_frozen_bilstm", "full_finetune", "lora", "lorma", "prefix_tuning", "prompt_tuning",
                       "knowledge_distillation") if n in RESULTS]
cands = {"all": MEMBERS}
ranked = sorted(MEMBERS, key=lambda n: -RESULTS[n]["dev"]["mean"])
for k in (2, 3, 4): cands[f"top{k}"] = ranked[:k]
sc = {k: eval_dev(ens_probs(v))["mean"] for k, v in cands.items()}
for k, v in sc.items(): print(f"{k:9s} {v:.4f}  {cands[k]}")
best_key = "all"
for k, v in sc.items():
    if v > sc[best_key] + 0.01: best_key = k
FINAL_MEMBERS = cands[best_key]; print("-> using ensemble:", best_key, FINAL_MEMBERS)
FINAL_PROBS = ens_probs(FINAL_MEMBERS)

best = (eval_dev(FINAL_PROBS)["mean"], 0.0, 0.0)
for tau, lam in [(0, .3), (0, .6), (0, 1), (.25, 0), (.5, 0), (.25, .3), (.25, .6), (.5, .6)]:
    s = eval_dev(FINAL_PROBS, make_decoder(tau, lam))["mean"]
    print(f"tau={tau} lam={lam}: {s:.4f}")
    if s > best[0] + 0.003: best = (s, tau, lam)
TAU, LAM = best[1], best[2]; DECODER = make_decoder(TAU, LAM)
FINAL = eval_dev(FINAL_PROBS, DECODER)
print(f"\nchosen tau={TAU}, lambda={LAM}")
print({k: round(v, 4) for k, v in FINAL.items()})

all       0.2251  ['E0_frozen_bilstm', 'full_finetune', 'lora', 'lorma', 'prefix_tuning', 'prompt_tuning', 'knowledge_distillation']
top2      0.2513  ['E0_frozen_bilstm', 'lorma']
top3      0.2455  ['E0_frozen_bilstm', 'lorma', 'prefix_tuning']
top4      0.2437  ['E0_frozen_bilstm', 'lorma', 'prefix_tuning', 'knowledge_distillation']
-> using ensemble: top2 ['E0_frozen_bilstm', 'lorma']
tau=0 lam=0.3: 0.2366
tau=0 lam=0.6: 0.2291
tau=0 lam=1: 0.2178
tau=0.25 lam=0: 0.2328
tau=0.5 lam=0: 0.1150
tau=0.25 lam=0.3: 0.2345
tau=0.25 lam=0.6: 0.2173
tau=0.5 lam=0.6: 0.2067

chosen tau=0.0, lambda=0.0
{'CL': 0.2674, 'CL_13': 0.2057, 'IT': 0.2353, 'IT_13': 0.1991, 'mean': 0.2513, 'mean_13': 0.2024, 'pooled': 0.2748, 'pooled_13': 0.2325}


## 13. Write the predictions and validate the format

In [ ]:
def predict_split_labels(split):
    pm = ens_probs(FINAL_MEMBERS)[split]; out = {}
    for d in DOCS[split]:
        labs = [id_to_label[int(i)] for i in DECODER(pm[d["id"]], d["domain"])]
        assert len(labs) == d["n"] == len(d["sents"]), d["id"]
        assert all(l in label_to_id for l in labs)
        out[d["id"]] = labs
    return out

sub = {}
for dom in ("CL", "IT"):
    p = predict_split_labels(dom + "_test"); sub[dom] = p
    json.dump(p, open(os.path.join(OUTPUT_DIR, f"prediction_{dom}.json"), "w"), ensure_ascii=False, indent=1)
overlap = set(sub["CL"]) & set(sub["IT"]); assert not overlap, f"CL/IT test ids overlap: {list(overlap)[:3]}"
combined = {**sub["CL"], **sub["IT"]}
pj = os.path.join(OUTPUT_DIR, "prediction.json"); json.dump(combined, open(pj, "w"), ensure_ascii=False, indent=1)
zp = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z: z.write(pj, arcname="prediction.json")
print("test docs:", len(combined), "| sentences:", sum(len(v) for v in combined.values()), "->", zp)
print("predicted label distribution:", Counter(l for v in combined.values() for l in v).most_common())

test docs: 10 | sentences: 2197 -> /content/drive/MyDrive/CS781/Capstone-Project/Phase-3-TXL-hier/submission.zip
predicted label distribution: [('RatioOfTheDecision', 958), ('Fact', 520), ('ArgumentPetitioner', 213), ('ArgumentRespondent', 210), ('Statute', 127), ('PrecedentReliedUpon', 108), ('RulingByPresentCourt', 43), ('None', 9), ('Issue', 8), ('RulingByLowerCourt', 1)]


## 14. Reading the results honestly

- **What should drive the gain over the original 0.25–0.27**: document context via the BiLSTM
  (compare `A0` → `A1` → `E0` in Section 9 -- that is your context ablation), class-balanced loss, and
  the ensemble.
- **0.60 is a target, not a guarantee.** This notebook has not been run against your live checkpoint/GPU.
  With only 40 training documents per domain and a 5-document dev set, Macro-F1 has roughly ±0.05 noise,
  and the strict-13-label score will sit below the sklearn-default one whenever a class is absent from
  dev (Dissent, PrecedentOverruled, PrecedentNotReliedUpon are the usual suspects). Check the per-class
  `classification_report` (add a cell like Section 11 of the GQLA notebook if you want it here too) to
  see which rare classes cap the macro average.
- **If you're still below target**, in rough order of expected payoff: (1) raise `WINDOW` toward 64 for
  more BiLSTM context per window (needs more GPU memory); (2) let `full_finetune`/KD train longer if dev
  is still improving when `PATIENCE` triggers; (3) add neighbouring-sentence text as extra input alongside
  the pooled vector; (4) try `prefix_tuning` with a larger `PREFIX_LEN` -- it's the one technique here that
  reuses the Transformer-XL memory mechanism directly, so it may have more headroom than generic
  LoRA/LoRMA on this architecture; (5) refit the chosen configuration on train+dev for the actual test
  submission once you've picked a winner.
- **LoRMA**: same formulation note as the original notebook -- verify the multiplicative-update
  formulation against the paper/official code before describing it in your report.
- **Fair-comparison note**: the capstone rules (Section 9.2) require the 6 experiments to differ in
  *adaptation technique*, not just hyperparameters -- that's satisfied here (full FT / LoRA / LoRMA /
  Prefix / Prompt / KD), and all six now share the *same* BiLSTM document head and training loop, so any
  gap between them in the results table is attributable to the adaptation technique rather than to the
  head architecture changing underneath them.